# NB10: Regression on aa-only features (corrected feature set)

**Context**: NB04 feature evaluation on contaminated labels selected `aa+kmer2`
(avg AUC=0.656). The corrected re-run (`scripts/run_feature_eval_corrected.py`,
2026-07-30) on org-filtered labels reverses the winner: `aa` (20 features) ties
`aa+physicochemical` at AUC=0.658, while `aa+kmer2` *drops* to 0.621 (Δ=−0.035).
Adding kmer2 features was a label-contamination artifact.

The original regression models in `data/models/stressor_*_regression.cbm` and
`data/regression_model_metrics.csv` were trained on `aa+kmer2` features via
`scripts/run_regression_only.py`. This notebook re-trains on `aa`-only (20 features)
to produce a methodologically clean pipeline.

**Outputs**
- `data/models/stressor_{name}_regression_aa.cbm` — 11 trained models
- `data/models/stressor_{name}_reg_predictions_aa.parquet` — test-set predictions
- `data/regression_model_metrics_aa.csv` — Spearman ρ, AUC-from-ranking per metal
- `figures/nb10_regression_aa_comparison.pdf` — comparison vs aa+kmer2 models

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR  = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
FIGS_DIR  = PROJ_ROOT / 'figures'
MODEL_DIR.mkdir(exist_ok=True)

SEED       = 42
TEST_SIZE  = 0.2
CB_PARAMS  = dict(iterations=500, learning_rate=0.05, depth=6,
                  loss_function='RMSE', random_seed=SEED, verbose=False)

METAL_STRESSORS = ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'Hg', 'Mn', 'Fe', 'Se', 'Al']

In [2]:
labeled_pd = pd.read_parquet(DATA_DIR / 'labeled_pd.parquet')
X_aa = pd.read_parquet(DATA_DIR / 'features_aa.parquet').drop(columns=['organism'], errors='ignore')
groups = labeled_pd['organism']

print(f'labeled_pd: {labeled_pd.shape}')
print(f'X_aa: {X_aa.shape}  (features: {list(X_aa.columns)})')
print(f'Organisms: {groups.nunique()}')

fit_cols_available = [f'{s}_fit' for s in METAL_STRESSORS if f'{s}_fit' in labeled_pd.columns]
print(f'Fitness columns available for metals: {[c.replace("_fit","") for c in fit_cols_available]}')

labeled_pd: (215051, 96)
X_aa: (215051, 20)  (features: ['aa_A', 'aa_C', 'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H', 'aa_I', 'aa_K', 'aa_L', 'aa_M', 'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S', 'aa_T', 'aa_V', 'aa_W', 'aa_Y'])
Organisms: 60
Fitness columns available for metals: ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'Hg', 'Mn', 'Fe', 'Se', 'Al']


In [3]:
def train_regression_aa(stressor):
    """Train CatBoostRegressor on aa-only features for one metal stressor.

    Proteins with NaN fitness are excluded — they had no experiment for this stressor.
    GroupShuffleSplit ensures organisms are split whole (no leakage).
    AUC_from_ranking: treating predicted fitness score as a ranking, measures
    how well it recovers binary positives (fitness < -2.0).
    """
    fit_col = f'{stressor}_fit'
    if fit_col not in labeled_pd.columns:
        log.warning(f'{stressor}: no {fit_col} column — skipping')
        return None

    mask    = labeled_pd[fit_col].notna()
    y       = labeled_pd.loc[mask, fit_col].astype(float)
    X       = X_aa[mask]
    g       = groups[mask]
    n_orgs  = g.nunique()
    n_tested = int(mask.sum())

    if n_orgs < 2:
        log.warning(f'{stressor}: only {n_orgs} org(s); need ≥2 to split — skipping')
        return None

    log.info(f'{stressor}: {n_tested:,} proteins across {n_orgs} organisms '
             f'({int((y < -2).sum())} binary positives)')

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    tr_idx, te_idx = next(gss.split(X, y, groups=g))
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    g_te       = g.iloc[te_idx]

    log.info(f'  train={len(X_tr):,} ({g.iloc[tr_idx].nunique()} orgs)  '
             f'test={len(X_te):,} ({g_te.nunique()} orgs)')

    model = CatBoostRegressor(**CB_PARAMS)
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    rho, pval = spearmanr(y_te, y_pred)
    rmse = float(np.sqrt(np.mean((y_te.values - y_pred) ** 2)))

    y_binary_te = (y_te < -2).astype(int)
    auc = float(roc_auc_score(y_binary_te, -y_pred)) if y_binary_te.sum() > 0 else None

    model.save_model(str(MODEL_DIR / f'stressor_{stressor}_regression_aa.cbm'))
    pd.DataFrame({
        'y_test': y_te.values, 'y_pred': y_pred, 'group': g_te.values,
    }).to_parquet(MODEL_DIR / f'stressor_{stressor}_reg_predictions_aa.parquet')

    return {
        'Spearman_rho': float(rho), 'Spearman_pval': float(pval),
        'RMSE': rmse, 'AUC_from_ranking': auc,
        'n_tested': n_tested, 'n_binary_pos': int((y < -2).sum()),
        'pos_rate': float((y < -2).mean()), 'n_orgs': int(n_orgs),
    }

In [4]:
reg_metrics_aa = {}
for s in METAL_STRESSORS:
    m = train_regression_aa(s)
    if m is not None:
        reg_metrics_aa[s] = m
        print(f'{s:4s}  rho={m["Spearman_rho"]:+.3f}  p={m["Spearman_pval"]:.1e}  '
              f'AUC={m["AUC_from_ranking"]:.3f}  n_orgs={m["n_orgs"]}')

df_aa = pd.DataFrame(reg_metrics_aa).T
df_aa.to_csv(DATA_DIR / 'regression_model_metrics_aa.csv')
print(f'\nSaved regression_model_metrics_aa.csv — {len(df_aa)} metals')

2026-08-03 22:14:20,041 INFO Zn: 142,269 proteins across 40 organisms (7300 binary positives)


2026-08-03 22:14:20,163 INFO   train=114,530 (32 orgs)  test=27,739 (8 orgs)


2026-08-03 22:14:25,549 INFO Cu: 148,953 proteins across 41 organisms (8101 binary positives)


2026-08-03 22:14:25,674 INFO   train=116,941 (32 orgs)  test=32,012 (9 orgs)


Zn    rho=+0.177  p=8.3e-195  AUC=0.694  n_orgs=40


...

Fe    rho=+0.142  p=2.9e-123  AUC=0.624  n_orgs=32


2026-08-03 22:14:55,659 INFO Al: 142,561 proteins across 39 organisms (6954 binary positives)


2026-08-03 22:14:55,767 INFO   train=110,731 (31 orgs)  test=31,830 (8 orgs)


Se    rho=+0.161  p=2.5e-40  AUC=0.677  n_orgs=7


Al    rho=+0.099  p=1.1e-69  AUC=0.633  n_orgs=39

Saved regression_model_metrics_aa.csv — 11 metals


## Comparison: aa-only vs aa+kmer2 regression

The original models in `regression_model_metrics.csv` used `aa+kmer2` features
(the contaminated-label winner). This section checks whether the 20-feature `aa`
model recovers similar or better AUC-from-ranking on the same task.

In [5]:
df_kmer = pd.read_csv(DATA_DIR / 'regression_model_metrics.csv', index_col=0)

# Align on common metals
common = df_aa.index.intersection(df_kmer.index)
df_cmp = pd.DataFrame({
    'AUC_aa':    df_aa.loc[common, 'AUC_from_ranking'],
    'AUC_kmer2': df_kmer.loc[common, 'AUC_from_ranking'],
    'rho_aa':    df_aa.loc[common, 'Spearman_rho'],
    'rho_kmer2': df_kmer.loc[common, 'Spearman_rho'],
    'n_orgs':    df_aa.loc[common, 'n_orgs'],
})
df_cmp['ΔAUC (aa − kmer2)'] = df_cmp['AUC_aa'] - df_cmp['AUC_kmer2']
df_cmp['Δrho (aa − kmer2)'] = df_cmp['rho_aa'] - df_cmp['rho_kmer2']

print('AUC-from-ranking and Spearman ρ comparison (aa-only vs aa+kmer2):')
print(df_cmp[['AUC_aa', 'AUC_kmer2', 'ΔAUC (aa − kmer2)',
              'rho_aa', 'rho_kmer2', 'Δrho (aa − kmer2)', 'n_orgs']].round(3).to_string())

n_better_auc = (df_cmp['ΔAUC (aa − kmer2)'] > 0).sum()
n_better_rho = (df_cmp['Δrho (aa − kmer2)'] > 0).sum()
print(f'\naa beats aa+kmer2 on AUC: {n_better_auc}/{len(df_cmp)} metals')
print(f'aa beats aa+kmer2 on rho:  {n_better_rho}/{len(df_cmp)} metals')

AUC-from-ranking and Spearman ρ comparison (aa-only vs aa+kmer2):
    AUC_aa  AUC_kmer2  ΔAUC (aa − kmer2)  rho_aa  rho_kmer2  Δrho (aa − kmer2)  n_orgs
Zn   0.694      0.688              0.006   0.177      0.195             -0.017    40.0
Cu   0.646      0.655             -0.008   0.173      0.198             -0.025    41.0
Cd   0.607      0.556              0.051   0.209      0.199              0.010     2.0
Co   0.668      0.679             -0.011   0.215      0.230             -0.015    41.0
Ni   0.661      0.663             -0.002   0.153      0.159             -0.006    42.0
Cr   0.695      0.671              0.025   0.142      0.141              0.001    11.0
Hg   0.779      0.774              0.006   0.154      0.175             -0.021     9.0
Mn   0.602      0.623             -0.021   0.003      0.050             -0.047     4.0
Fe   0.624      0.627             -0.003   0.142      0.148             -0.006    32.0
Se   0.677      0.699             -0.022   0.161      0.155     

In [6]:
metals_sorted = df_aa['AUC_from_ranking'].sort_values(ascending=False).index

fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Left: AUC-from-ranking by metal, aa vs aa+kmer2
ax = axs[0]
x = np.arange(len(metals_sorted))
w = 0.35
auc_aa    = [df_cmp.loc[m, 'AUC_aa']    if m in df_cmp.index else np.nan for m in metals_sorted]
auc_kmer2 = [df_cmp.loc[m, 'AUC_kmer2'] if m in df_cmp.index else np.nan for m in metals_sorted]
ax.bar(x - w/2, auc_aa,    width=w, label='aa (20 feat)',       color=PALETTE[0], edgecolor='k', lw=0.5)
ax.bar(x + w/2, auc_kmer2, width=w, label='aa+kmer2 (420 feat)',color=PALETTE[1], edgecolor='k', lw=0.5)
ax.axhline(0.5, color='gray', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels(metals_sorted, rotation=45, ha='right')
ax.set_xlabel('Metal stressor')
ax.set_ylabel('AUC-from-ranking (test set)')
ax.set_title('AUC: aa vs aa+kmer2')
ax.legend(fontsize=8)
grid_h(ax)

# Right: ΔAUC (aa − aa+kmer2)
ax = axs[1]
delta = [df_cmp.loc[m, 'ΔAUC (aa − kmer2)'] if m in df_cmp.index else 0 for m in metals_sorted]
colors = [PALETTE[0] if d >= 0 else PALETTE[1] for d in delta]
ax.bar(metals_sorted, delta, color=colors, edgecolor='k', lw=0.5)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Metal stressor')
ax.set_ylabel('ΔAUC (aa − aa+kmer2)')
ax.set_title('aa improvement over aa+kmer2')
ax.tick_params(axis='x', rotation=45)
grid_h(ax)

fig.suptitle('NB10: Regression AUC-from-ranking — aa-only vs aa+kmer2 features', y=1.02)
save(fig, FIGS_DIR / 'nb10_regression_aa_comparison')
print('Figure saved.')

Figure saved.


In [7]:
print('=== NB10 Summary ===')
print(f'Trained aa-only regression models for {len(df_aa)} metals.')
print(f'\naa-only model metrics (sorted by AUC-from-ranking):')
print(df_aa[['Spearman_rho', 'Spearman_pval', 'AUC_from_ranking', 'n_orgs', 'n_tested']]
      .sort_values('AUC_from_ranking', ascending=False)
      .round(3).to_string())

print(f'\nMean AUC-from-ranking (aa-only): {df_aa["AUC_from_ranking"].mean():.3f}')
if not df_kmer.empty:
    kmer_common = df_kmer.loc[df_aa.index, 'AUC_from_ranking']
    print(f'Mean AUC-from-ranking (aa+kmer2): {kmer_common.mean():.3f}')
    print(f'Mean ΔAUC (aa − aa+kmer2): {(df_aa["AUC_from_ranking"] - kmer_common).mean():+.3f}')

=== NB10 Summary ===
Trained aa-only regression models for 11 metals.

aa-only model metrics (sorted by AUC-from-ranking):
    Spearman_rho  Spearman_pval  AUC_from_ranking  n_orgs  n_tested
Hg         0.154          0.000             0.779     9.0   29263.0
Cr         0.142          0.000             0.695    11.0   35693.0
Zn         0.177          0.000             0.694    40.0  142269.0
Se         0.161          0.000             0.677     7.0   19702.0
Co         0.215          0.000             0.668    41.0  150081.0
Ni         0.153          0.000             0.661    42.0  152937.0
Cu         0.173          0.000             0.646    41.0  148953.0
Al         0.099          0.000             0.633    39.0  142561.0
Fe         0.142          0.000             0.624    32.0  115488.0
Cd         0.209          0.000             0.607     2.0    6184.0
Mn         0.003          0.873             0.602     4.0   14370.0

Mean AUC-from-ranking (aa-only): 0.662
Mean AUC-from-ranking